# Load libraries

In [ ]:
from zzzs_util import *

import os
import gc
import ctypes
import joblib
from tqdm.auto import tqdm

libc = ctypes.CDLL("libc.so.6")

import pytz
import cudf
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

from sklearn.preprocessing import RobustScaler

In [ ]:
print(cudf.__version__)

In [ ]:
COMP_LOC = "/kaggle/input/child-mind-institute-detect-sleep-states/"
DATA_LOC = "/kaggle/input/zzzs-seriesfolds-test/"

NUM_FOLDS = 7

# Custom functions

In [ ]:
def data_cleaning(data):
    
    # Data cleaning
    data = data.fillna(0)
    data = data.astype(float)
    data = data.to_numpy()
    
    return data

In [ ]:
def generate_target_array(data_length, targets):
    """
    Generates a binary target numpy array based on the provided targets.
    
    Parameters:
    - data_length (int): Length of the data series.
    - targets (list of tuples): Each tuple is a pair of start and end indices for sleep events.
    
    Returns:
    - numpy array: Binary target array where 1 indicates awake and 0 indicates sleep.
    """
    y = np.ones(data_length, dtype=int)  # Start with an array of ones (awake)

    for start, end in targets:
        y[int(start):int(end)+1] = 0  # Set the indices corresponding to sleep events to 0

    return y.reshape(-1, 1)

# Create [fold]ers

In [ ]:
# Define the base path where you want to create the folders
base_path = './'

# List of folder names
folder_names = [f'fold{i}' for i in range(1,NUM_FOLDS+1)]

# Loop through the list and create each folder
for folder in folder_names:
    folder_path = os.path.join(base_path, folder)
    
    # Check if the folder already exists to avoid throwing an error
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# Events data

In [ ]:
PATH = COMP_LOC + "train_events.csv"

events_df = pd.read_csv(PATH)

events_df = events_df[events_df['step'].notnull()]
events_df.drop_duplicates(inplace=True)
events_df = events_df.reset_index(drop=True)

print(events_df.shape)
events_df.head()

# NumpyFolds

In [ ]:
foldrun = 1

for fold in tqdm(range(1,NUM_FOLDS+1)):
    
    if fold != foldrun: continue
    
    FOLD_PATH = DATA_LOC + f"fold{fold}/"
    
    for file in tqdm(os.listdir(FOLD_PATH)):
        
        
        ############
        # SERIES ID
        ############
        
        series_id = file.split('.parquet')[0]
        TRUNC_SERIES = False
        
        # Load the data with cuDF
        PATH = FOLD_PATH + f"{series_id}.parquet"
        data = cudf.read_parquet(PATH)
        data = data.reset_index(drop=True)
        
        ############
        # Target
        ############
        
        targets = []
        events = events_df[events_df.series_id == series_id]
        
#         events['night_diff'] = events['night'].diff()
#         if events['night_diff'].max() > 1:
#             TRUNC_SERIES = True
#         events.drop(columns=['night_diff'], inplace=True)
        
            
        trunc_data = pd.DataFrame()
        for i in range(len(events)-1):
            if events.iloc[i].event =='onset' and events.iloc[i+1].event =='wakeup' and events.iloc[i].night==events.iloc[i+1].night:
                start,end = events.timestamp.iloc[i],events.timestamp.iloc[i+1]
                
#                 if TRUNC_SERIES:
#                     # Insert logic to only day, day+1 from series where start occurs
#                     pass
                    

                start_id = data.loc[data.timestamp ==start].index.values[0]
                end_id = data.loc[data.timestamp ==end].index.values[0]
                targets.append((start_id,end_id))
                
        
        ############
        # Features
        ############
        
        # Feature Engineering
        data = feature_engineering(data)
        
        free_memory()
        
        # Scaling
#         features_to_scale = [column for column in data.columns if column not in ['series_id', 'step', 'timestamp']]
#         scaler = RobustScaler()
#         data[features_to_scale] = scaler.fit_transform(data[features_to_scale].to_pandas())
        
        
        # Final features
        data.drop(columns=['series_id','timestamp'], inplace=True)
#         np.save(f"./fold{fold}/{series_id}.npy", data.to_pandas().values)

        data = data.to_pandas()
        free_memory()

        data = data_cleaning(data)
        free_memory()

#         print("Data cleaned")

        y = generate_target_array(data.shape[0], targets)
        free_memory()

#         print("Target data generated")
        
        scaler = joblib.load(f"/kaggle/input/zzzs-stdscalerfit/scaler_fold{fold}.pkl")

        # Scale the data
        X = scaler.transform(data)
        free_memory()

#         print("Scaler transform complete")


#         joblib.dump((targets, data, series_id), f"./fold{fold}/{series_id}.pkl")
        
        joblib.dump((X, y, series_id), f"./fold{fold}/{series_id}.pkl")
        
        

In [ ]:
# X, y, series_id = joblib.load(f"./fold1/29c75c018220.pkl")

# print(X.shape)
# print(y.shape)
# print(series_id)

# References

In [ ]:
# https://www.kaggle.com/code/werus23/sleep-critical-point-prepare-data